# ???? Vietnam Stock Market Analysis & Machine Learning Forecasting

**Author:** Vu Thanh Phong ([@PhongPro69](https://github.com/PhongPro69))  
**Objective:** End-to-end quantitative exploratory data analysis (EDA), technical feature engineering (RSI, MACD, Bollinger Bands, ATR), predictive machine learning modeling (XGBoost, Random Forest, ARIMA), and strategy backtesting for Vietnam Equities (VN30).

## 1. Setup & Environment

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data_loader import VNStockDataLoader, VN30_TICKERS, TICKER_NAMES
from src.feature_engineering import FeatureEngineer
from src.models.ml_forecaster import TreeEnsembleForecaster
from src.models.baseline import ARIMABaseline
from src.models.evaluator import ModelEvaluator

## 2. Ingesting Financial Data (FPT - Technology Leader in VN30)

In [ ]:
loader = VNStockDataLoader(cache_dir='../data')
df_fpt = loader.fetch_data('FPT', start_date='2021-01-01')
print(f'Total trading days loaded: {len(df_fpt)}')
df_fpt.tail()

## 3. Quantitative Feature Engineering
We construct 30+ non-lookahead features: Trend (SMA 20/50/200), Momentum (RSI 14, MACD), Volatility (Bollinger Bands, ATR), and Temporal Lag Returns.

In [ ]:
fe = FeatureEngineer()
df_tech = fe.add_technical_indicators(df_fpt)
X, y, feature_cols = fe.prepare_modeling_data(df_fpt, target_col='close')
print(f'Engineered matrix shape: {X.shape}, Target length: {len(y)}')
X.head()

## 4. Chronological Train-Test Split & ML Modeling
Financial time-series data must **never** be randomly shuffled. We apply a strict chronological train/test split.

In [ ]:
split_ratio = 0.8
split_idx = int(len(X) * split_ratio)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Fit Models
xgb = TreeEnsembleForecaster(model_type='xgboost', n_estimators=100, learning_rate=0.05)
xgb.fit(X_train, y_train)
preds_xgb = xgb.predict(X_test)

rf = TreeEnsembleForecaster(model_type='random_forest', n_estimators=150)
rf.fit(X_train, y_train)
preds_rf = rf.predict(X_test)

# Evaluate
metrics_xgb = ModelEvaluator.calculate_metrics(y_test.values, preds_xgb)
metrics_rf = ModelEvaluator.calculate_metrics(y_test.values, preds_rf)

pd.DataFrame([metrics_xgb, metrics_rf], index=['XGBoost', 'Random Forest'])

## 5. Strategy Backtesting & Risk-Adjusted Returns

In [ ]:
bt = ModelEvaluator.backtest_strategy(
    prices=y_test,
    predicted_prices=pd.Series(preds_xgb, index=y_test.index),
    threshold=0.003,
    initial_capital=100_000_000.0
)
print(f"Strategy Total Return: {bt['total_return_strategy_pct']:+.2f}%")
print(f"Benchmark Buy & Hold Return: {bt['total_return_benchmark_pct']:+.2f}%")
print(f"Sharpe Ratio: {bt['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {bt['max_drawdown_pct']:.2f}%")